In [13]:
import numpy as np
import pandas as pd
import psycopg
from sqlalchemy import create_engine
import os
import dotenv
dotenv.load_dotenv()
POSTGRES_PASSWORD = os.getenv('POSTGRES_PASSWORD')

In [14]:
dbms = 'postgresql'
user = 'postgres'
package = 'psycopg'
password = POSTGRES_PASSWORD
host = 'localhost'
port = '5432'
db = 'contrans'

engine = create_engine(f'{dbms}+{package}://{user}:{password}@{host}:{port}/{db}')

In [15]:
myquery = '''
    SELECT *
    FROM members
'''

pd.read_sql_query(myquery, con=engine)

,bioguide_id,full_name,chamber,state,party,district,birthyear,image,office_address,phone,website,left_right_ideology
0,A000055,Robert B. Aderholt,House of Representatives,Alabama,Republican,4.0,1965.0,https://www.congress.gov/img/member/a000055_20...,"272 Cannon House Office Building, Washington, ...",(202) 225-4876,https://aderholt.house.gov/,0.405
1,A000148,Jake Auchincloss,House of Representatives,Massachusetts,Democratic,4.0,1988.0,https://www.congress.gov/img/member/67817e391f...,"1524 Longworth House Office Building, Washingt...",(202) 225-5931,https://auchincloss.house.gov,-0.272
2,A000369,Mark E. Amodei,House of Representatives,Nevada,Republican,2.0,1958.0,https://www.congress.gov/img/member/a000369_20...,"104 Cannon House Office Building, Washington, ...",(202) 225-6155,https://amodei.house.gov,0.384
3,A000370,Alma S. Adams,House of Representatives,North Carolina,Democratic,12.0,1946.0,https://www.congress.gov/img/member/a000370_20...,"2436 Rayburn House Office Building, Washington...",(202) 225-1510,https://adams.house.gov,-0.464
4,A000371,Pete Aguilar,House of Representatives,California,Democratic,33.0,1979.0,https://www.congress.gov/img/member/a000371_20...,"108 Cannon House Office Building, Washington, ...",(202) 225-3201,https://aguilar.house.gov/,-0.328
...,...,...,...,...,...,...,...,...,...,...,...,...
550,W000831,James R. Walkinshaw,House of Representatives,Virginia,Democratic,11.0,1982.0,https://www.congress.gov/img/member/68c1bd4ca9...,"2265 Rayburn House Office Building, Washington...",(202) 225-1492,https://walkinshaw.house.gov/,-0.428
551,W000832,Aisha Wahab,House of Representatives,California,Democratic,14.0,1987.0,https://www.congress.gov/img/member/6a986b632a...,"174 Cannon House Office Building, Washington, ...",(202) 225-5065,https://wahab.house.gov/,NaN
552,Y000064,Todd Young,Senate,Indiana,Republican,NaN,1972.0,https://www.congress.gov/img/member/y000064_20...,185 Dirksen Senate Office Building Washington...,(202) 224-5623,https://www.young.senate.gov,0.444
553,Y000067,Rudy Yakym III,House of Representatives,Indiana,Republican,2.0,1984.0,https://www.congress.gov/img/member/y000067_20...,"349 Cannon House Office Building, Washington, ...",(202) 225-3915,https://yakym.house.gov,0.521


SQL Etiquette:

1. Clauses (commands that operate on the data) are CAPS
2. Data tables and columns are lowercase
3. Every clause on a new line
4. If a clause is connected or an extension of a previous clause, indent it!

In [20]:
name = 'John J. McGuire III'

myquery = f'''
SELECT bioguide_id
FROM members
WHERE full_name = '{name}'
'''

pd.read_sql_query(myquery, con=engine)

,bioguide_id
0,M001239


In [39]:
### What are top organizations spending for and against a given member of Congress?

name = 'John J. McGuire III'
bioguide_id = 'M001239'

myquery = f'''
    SELECT 
        c.bioguide_id,
        e.expenditure_amount,
        e.committee_name,
        e.committee_type_full,
        e.support_oppose_indicator
    FROM expenditures e
    INNER JOIN crosswalk c
        ON e.fec_id = c.fec_id
    WHERE c.bioguide_id = '{bioguide_id}'
'''
pd.read_sql_query(myquery, con=engine)

,bioguide_id,expenditure_amount,committee_name,committee_type_full,support_oppose_indicator
0,M001239,1645.34,NO GOING BACK PAC INC.,Super PAC (Independent Expenditure-Only),S
1,M001239,5361.36,VIRGINIANS FOR CONSERVATIVE LEADERSHIP PAC,Super PAC (Independent Expenditure-Only),S
2,M001239,84243.45,NO GOING BACK PAC INC.,Super PAC (Independent Expenditure-Only),S
3,M001239,18117.97,CIVIC RENEWAL FUND INC.,Super PAC (Independent Expenditure-Only),S
4,M001239,2165.02,CIVIC RENEWAL FUND INC.,Super PAC (Independent Expenditure-Only),S
5,M001239,850.28,NO GOING BACK PAC INC.,Super PAC (Independent Expenditure-Only),S
6,M001239,42121.73,NO GOING BACK PAC INC.,Super PAC (Independent Expenditure-Only),S
7,M001239,32000.00,OUR VIRGINIA,Super PAC (Independent Expenditure-Only),O
8,M001239,53280.00,OUR VIRGINIA,Super PAC (Independent Expenditure-Only),O
9,M001239,5613.20,VIRGINIANS FOR CONSERVATIVE LEADERSHIP PAC,Super PAC (Independent Expenditure-Only),S


In [75]:


name = 'John J. McGuire III'
bioguide_id = 'M001239'

myquery = f'''
    SELECT
        SUM(e.expenditure_amount)
            AS expenditure_amount,
        e.support_oppose_indicator
    FROM expenditures e
    INNER JOIN crosswalk c
        ON e.fec_id = c.fec_id
    WHERE c.bioguide_id = '{bioguide_id}'
    GROUP BY e.support_oppose_indicator
'''
pd.read_sql_query(myquery, con=engine)

,expenditure_amount,support_oppose_indicator
0,1211210.64,O
1,411556.72,S


In [100]:
# Who are the most similar members obased on voting, and how do those members measure up ideologically?

myquery = '''
    SELECT
        e.committee_name,
        SUM(m.left_right_ideology * e.expenditure_amount * 
        CASE
            WHEN e.support_oppose_indicator = 'S' THEN 1
            WHEN e.support_oppose_Indicator = 'O' THEN -1
        END) / SUM(e.expenditure_amount) AS ideology
    FROM expenditures e
    INNER JOIN crosswalk c
        ON e.fec_id = c.fec_id
    INNER JOIN members m
        ON c.bioguide_id = m.bioguide_id
    GROUP BY e.committee_name, m.bioguide_id
'''
df = pd.read_sql_query(myquery, con=engine)

In [102]:
df.sort_values('ideology')

,committee_name,ideology
1115,WE THE PEOPLE...CAN'T EVEN,-0.972
638,SECURITY IS STRENGTH PAC,-0.856
56,PALMETTO ACTION,-0.856
412,INVEST IN TOMORROW COALITION PAC,-0.856
906,HOLD THE LINE PAC,-0.856
...,...,...
933,ALAMEDA COUNTY DEMOCRATIC CENTRAL COMMITTEE,NaN
1041,UNITED DEMOCRACY PROJECT ('UDP'),NaN
1199,UNITY & JUSTICE FUND,NaN
1208,BOLD AMERICA,NaN
